In [ ]:
%pip install pandas matplotlib seaborn xlrd

Por que eu precisei fazer isso?? Não tava dando de jeito nenhum só dar o pip

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")


In [ ]:
path = r"Tabela 1.1.1.xls"

In [ ]:
indicador_1 = pd.read_excel(path, engine="xlrd")

In [ ]:
indicador_1


In [ ]:

bruto = pd.read_excel(path, engine="xlrd", header=None)

indicador_1 = bruto.iloc[8:41].copy()
indicador_1.columns = [
    "uf_regiao",
    "total",
    "total_branca",
    "total_preta_parda",
    "homem_branca",
    "homem_preta_parda",
    "mulher_branca",
    "mulher_preta_parda",
]
indicador_1 = indicador_1.reset_index(drop=True)


for col in indicador_1.columns[1:]:
    indicador_1[col] = pd.to_numeric(indicador_1[col], errors="coerce")

indicador_1


In [ ]:
media_nacional = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil", "total"].item()
media_nacional

Qual(is) estado(s) está acima da média nacional de afazeres domésticos?

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]

acima_media = estados.loc[
    estados["total"] > media_nacional,
    ["uf_regiao", "total"],
].sort_values("total", ascending=False)

print(f"Média nacional (Brasil): {media_nacional:.2f} horas/semana")
print(f"Estados acima da média: {len(acima_media)}")
acima_media

Quais estados estão abaixo da média?

In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]

estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)]

abaixo_media = estados.loc[
    estados["total"] < media_nacional,
    ["uf_regiao", "total"],
].sort_values("total", ascending=False)

print(f"Média nacional (Brasil): {media_nacional:.2f} horas/semana")
print(f"Estados abaixo da média: {len(abaixo_media)}")
abaixo_media

Qual é o percentual de diferença no Brasil que mulheres trabalham mais que homens em casa?

In [ ]:

brasil = indicador_1.loc[indicador_1["uf_regiao"] == "Brasil"].iloc[0]

homem = (brasil["homem_branca"] + brasil["homem_preta_parda"]) / 2
mulher = (brasil["mulher_branca"] + brasil["mulher_preta_parda"]) / 2

# quanto % a mais as mulheres trabalham em relação aos homens
percentual = (mulher - homem) / homem * 100

print(f"Homens (média das cores):   {homem:.2f} h/semana")
print(f"Mulheres (média das cores): {mulher:.2f} h/semana")
print(f"Mulheres trabalham {percentual:.1f}% a mais que homens em casa, no Brasil.")

# recorte por cor (a tabela traz isso de fato)
perc_branca = (brasil["mulher_branca"] / brasil["homem_branca"] - 1) * 100
perc_preta_parda = (brasil["mulher_preta_parda"] / brasil["homem_preta_parda"] - 1) * 100
print(f"Branca:        mulheres +{perc_branca:.1f}%")
print(f"Preta ou parda: mulheres +{perc_preta_parda:.1f}%")


Agora responda:
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas em Minas Gerais?
- Quantas horas mulheres pretas trabalham a mais que homens brancos no Brasil?
- Quantas horas mulheres pretas trabalham a mais que mulheres brancas no Amapá?


In [ ]:
mg = indicador_1.loc[indicador_1["uf_regiao"] == "Minas Gerais"].iloc[0]

dif_horas_mg = mg["mulher_preta_parda"] - mg["mulher_branca"]
print(f"Em MG, mulheres pretas trabalham {dif_horas_mg:.2f} horas a mais que mulheres brancas.")
dif_horas_br = brasil["mulher_preta_parda"] - brasil["homem_branca"]
print(f"No Brasil, mulheres pretas trabalham {dif_horas_br:.2f} horas a mais que homens brancos.")
ap = indicador_1.loc[indicador_1["uf_regiao"] == "Amapá"].iloc[0]

dif_horas_ap = ap["mulher_preta_parda"] - ap["mulher_branca"]
print(f"No Amapá, mulheres pretas ou pardas trabalham {abs(dif_horas_ap):.2f} horas a menos que mulheres brancas.")

## Gráfico 1 — Barras horizontais por UF


In [ ]:
regioes = ["Norte", "Nordeste", "Sudeste", "Sul", "Centro-Oeste"]
estados = indicador_1[~indicador_1["uf_regiao"].isin(["Brasil"] + regioes)].copy()

fig, ax = plt.subplots(figsize=(9, 8))
ordem = estados.sort_values("total", ascending=False)
sns.barplot(data=ordem, y="uf_regiao", x="total", ax=ax, color="steelblue")
ax.axvline(
    media_nacional,
    color="crimson",
    linestyle="--",
    label=f"Média Brasil ({media_nacional:.1f} h)",
)
ax.set_title("Horas semanais de afazeres domésticos por UF")
ax.set_xlabel("Horas / semana")
ax.set_ylabel("UF")
ax.legend()
plt.tight_layout()
plt.show()


## Gráfico 2 — Barras agrupadas por região e sexo

Conforme o enunciado, homens e mulheres são calculados pela média simples das duas categorias de cor. Essa média não é ponderada pela população de cada grupo.


In [ ]:
reg = indicador_1[indicador_1["uf_regiao"].isin(regioes)].copy()
reg["homens"] = (reg["homem_branca"] + reg["homem_preta_parda"]) / 2
reg["mulheres"] = (reg["mulher_branca"] + reg["mulher_preta_parda"]) / 2
reg_long = reg.melt(
    id_vars="uf_regiao",
    value_vars=["homens", "mulheres"],
    var_name="sexo",
    value_name="horas",
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=reg_long, x="uf_regiao", y="horas", hue="sexo", ax=ax)
ax.set_title("Afazeres domésticos por região e sexo")
ax.set_xlabel("Região")
ax.set_ylabel("Horas / semana")
plt.tight_layout()
plt.show()


## Gráfico 3 — Histograma e box plot das horas por UF


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(estados["total"], bins=8, kde=True, ax=axes[0])
axes[0].axvline(media_nacional, color="crimson", linestyle="--")
axes[0].set_title("Frequência das horas médias por UF")
axes[0].set_xlabel("Horas / semana")
axes[0].set_ylabel("Número de UFs")

sns.boxplot(y=estados["total"], ax=axes[1])
axes[1].axhline(media_nacional, color="crimson", linestyle="--")
axes[1].set_title("Box plot das horas por UF")
axes[1].set_ylabel("Horas / semana")
plt.tight_layout()
plt.show()


## Gráfico 4 — Box plot por sexo e cor


In [ ]:
box_df = estados[
    ["homem_branca", "homem_preta_parda", "mulher_branca", "mulher_preta_parda"]
].melt(var_name="grupo", value_name="horas")

fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=box_df, x="grupo", y="horas", ax=ax)
ax.set_title("Distribuição das horas por sexo e cor (UFs)")
ax.set_xlabel("Grupo")
ax.set_ylabel("Horas / semana")
plt.xticks(rotation=15)
plt.tight_layout()
plt.show()
